# 006 — Probabilidad, incertidumbre y estadística básica

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

La probabilidad es el cálculo de la incertidumbre (axiomas de Kolmogórov: P(A) ≥ 0,
P(Ω) = 1, aditividad en disjuntos). Sobre ellos:

```text
condicional:    P(A|B) = P(A∩B) / P(B)
producto:       P(A∩B) = P(A|B) P(B)
independencia:  P(A∩B) = P(A) P(B)      (supuesto de modelado, no default)
Bayes:          P(H|E) = P(E|H) P(H) / P(E)    con  P(E) = P(E|H)P(H) + P(E|¬H)P(¬H)
esperanza:      E[X] = Σ x P(X=x)   ·   Var(X) = E[(X−E[X])²]
```

La **ley de los grandes números** justifica estimar probabilidades simulando (Monte Carlo)
— lo que hace el laboratorio con semilla fija.

**Ejemplo trabajado (test diagnóstico):** prevalencia 1 %, sensibilidad 90 %, falsos
positivos 9 %. P(enfermo|positivo) = 0.009/0.0981 ≈ **9 %**, no 90 %: el prior domina.
Traducción a IA: un clasificador "90 % preciso" sobre una clase rara produce sobre todo
falsas alarmas — accuracy sin tasa base no informa.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** De 10 000 archivos: 50 malware → 47.5 alarmas verdaderas; 9 950 limpios →
199 falsas alarmas. P(malware|alarma) = 47.5/(47.5+199) ≈ **0.193**. Aproximadamente 81 de
cada 100 alarmas son falsas, pese al "95 % de detección": la tasa base (0.5 %) domina.

**Ejercicio 2.** E[X] = (1+2+3+4+5+6)/6 = 3.5. E[X²] = 91/6 ≈ 15.17, Var = 15.17 − 12.25 ≈
**2.92** (σ ≈ 1.71). Pagar 4 € > 3.5 esperado → pérdida esperada de 0.5 €/tirada: no es
racional a largo plazo. Para una sola tirada la esperanza sigue siendo el criterio racional
de referencia, pero la aversión al riesgo puede justificar decisiones distintas — eso es
utilidad, no probabilidad.

**Ejercicio 3.** Las tres estimaciones deben variar poco alrededor del mismo valor; el
rango refleja error muestral. Si una semilla diera un valor radicalmente distinto,
sospecharíamos bug o n demasiado pequeño, no "otra distribución".

**Ejercicio 4.** Binomial(n, p), media np y desviación √(np(1−p)). La corrida debe caer a
~2σ de np; si no, la predicción o el supuesto sobre el laboratorio estaban mal — y eso es
información (clase 008).

In [ ]:
result = run_lab("probability", seed=6)
assert result["kind"] == "probability"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — Bayes verificado
malware = 10000 * 0.005          # 50
limpios = 10000 - malware        # 9950
verdaderas = malware * 0.95      # 47.5
falsas = limpios * 0.02          # 199
p = verdaderas / (verdaderas + falsas)
print(f"P(malware|alarma) = {p:.3f}; falsas por cada 100 alarmas ≈ {100*(1-p):.0f}")
assert abs(p - 0.1927) < 0.001

In [ ]:
# Ejercicio 3 — estabilidad entre semillas
valores = []
for s in (1, 2, 3):
    r = run_lab("probability", seed=s)
    valores.append(r)
    print(f"seed={s} →", {k: v for k, v in r.items() if k not in ('evidence', 'limitations')})
# Criterio: las estimaciones centrales varían dentro del error muestral esperado;
# la estructura (claves, kind) es idéntica en las tres corridas.

## Reflexión

1. El laboratorio `probability` estima frecuencias con una semilla fija. ¿Qué garantiza la
   ley de los grandes números sobre esas estimaciones y qué NO garantiza sobre una corrida
   con n pequeño?
2. Da un caso de tu contexto donde confundir P(A|B) con P(B|A) llevaría a una decisión
   equivocada, y calcula con números inventados pero coherentes cuánto cambia la respuesta.
3. Si duplicas el número de muestras del laboratorio, ¿qué esperas que pase con la media
   estimada y con su dispersión entre semillas? ¿Por qué?